In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

options = webdriver.ChromeOptions()
options.add_argument('--headless')  # Remove this line if you want to see the browser

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
url = "https://www.tender247.com/tenders?state=west+bengal&advancesearch=true"
driver.get(url)
time.sleep(5)

# Infinite scroll to load more tenders
SCROLL_PAUSE = 3
last_height = driver.execute_script("return document.body.scrollHeight")
for _ in range(10):  # Increase this range for more scrolling if needed
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(SCROLL_PAUSE)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

# Find all tender blocks
tender_blocks = driver.find_elements(By.CSS_SELECTOR, "div.bg-\\[\\#fff\\].hover\\:shadow-md")

print(f"Found {len(tender_blocks)} tenders.")

data = []

for tb in tender_blocks:
    try:
        text = tb.text
        # Parse using text lines as fallback if direct selector fails
        t247_id = ""
        bid_value = ""
        emd = ""
        closing_date = ""
        description = ""
        location = ""
        link = ""

        # T247 ID and Description
        try:
            t247_id = tb.find_element(By.XPATH, ".//span[contains(text(), 'T247 ID-')]").find_element(By.XPATH, '..').text
        except:
            t247_id = ""
        try:
            bid_value = tb.find_element(By.XPATH, ".//span[contains(text(), 'Bid Value:')]/following-sibling::div").text
        except:
            bid_value = ""
        try:
            emd = tb.find_element(By.XPATH, ".//span[contains(text(), 'EMD:')]/following-sibling::div").text
        except:
            emd = ""
        try:
            closing_date = tb.find_element(By.XPATH, ".//div[contains(@class,'float-left') and contains(text(), 'Left')]").text
        except:
            closing_date = ""
        try:
            description = tb.find_element(By.CSS_SELECTOR, "p > a > span").text
        except:
            description = ""
        try:
            location = tb.find_element(By.XPATH, ".//h3[contains(@class,'font-bold capitalize') and contains(text(),'India')]").text
        except:
            location = ""
        try:
            link_tag = tb.find_element(By.XPATH, ".//a[contains(text(),'BID NOW')]")
            link = link_tag.get_attribute("href")
        except:
            link = ""

        data.append({
            "T247 ID": t247_id,
            "Bid Value": bid_value,
            "EMD": emd,
            "Closing Date": closing_date,
            "Description": description,
            "Location": location,
            "Bid Link": link
        })
    except Exception as e:
        print("Error:", e)
        continue

driver.quit()

df = pd.DataFrame(data)
df.to_excel("tender247_west_bengal_scraped.xlsx", index=False)
print("Scraping done! Output: tender247_west_bengal_scraped.xlsx")
